# Chapter 08. 파이썬 모듈과 패키지

이 노트북은 [Chapter 08 원본 문서](../doc/Chapter%2008.%20%ED%8C%8C%EC%9D%B4%EC%8D%AC%20%EB%AA%A8%EB%93%88%EA%B3%BC%20%ED%8C%A8%ED%82%A4%EC%A7%80.md)를 실습용으로 변환한 자료입니다.

모듈, import, `__name__`, `sys.path`, 패키지, 절대 import, 상대 import를 코인 거래 예제로 연습합니다.

## 1. 실습 환경 준비

노트북 안에 임시 실습 디렉터리와 모듈 파일을 만들어 실제 `import` 동작을 확인합니다. 실습 파일은 프로젝트의 기존 소스 파일과 분리됩니다.

In [1]:
from pathlib import Path
import sys
import tempfile

lab_dir = Path(tempfile.mkdtemp(prefix="chapter08_"))
sys.path.insert(0, str(lab_dir))
print("실습 디렉터리:", lab_dir)
print("Python 실행 파일:", sys.executable)

실습 디렉터리: C:\Users\juyoung\AppData\Local\Temp\chapter08_802mun4p
Python 실행 파일: c:\Users\juyoung\AppData\Local\Programs\Python\Python313\python.exe


## 2. 모듈 만들기와 `import`

모듈은 Python 정의를 담은 `.py` 파일입니다. 가격 계산 함수를 모듈로 분리한 뒤 다른 코드에서 불러옵니다.

In [ ]:
import importlib

(lab_dir / "coin_utils.py").write_text(
    "def calculate_return(buy_price, sell_price):\n"
    "    return (sell_price - buy_price) / buy_price\n"
    "\n"
    "def calculate_fee(amount, fee_rate=0.0004):\n"
    "    return amount * fee_rate\n",
    encoding="utf-8",
)

coin_utils = importlib.import_module("coin_utils")
return_rate = coin_utils.calculate_return(100, 110)
fee = coin_utils.calculate_fee(10_000)
print("수익률:", return_rate)
print("수수료:", fee)

수익률: 0.1
수수료: 4.0


## 3. 함수 직접 import와 별칭

`from ... import ...`는 필요한 이름만 가져옵니다. `as`를 사용하면 모듈이나 함수에 별칭을 붙일 수 있습니다.

In [ ]:
calculate_fee = coin_utils.calculate_fee
calculate_return = coin_utils.calculate_return
utils = coin_utils
get_return = coin_utils.calculate_return

print(calculate_return(100, 105))
print(utils.calculate_fee(20_000))
print(get_return(200, 220))

0.05
8.0
0.1


## 4. `__name__ == "__main__"`

모듈을 직접 실행할 때만 테스트 코드가 실행되도록 `__name__` 조건을 사용합니다. import할 때는 조건 아래 코드가 실행되지 않습니다.

In [ ]:
import importlib

script_path = lab_dir / "fibo.py"
script_path.write_text(
    "def fib2(limit):\n"
    "    result = []\n"
    "    first, second = 0, 1\n"
    "    while first < limit:\n"
    "        result.append(first)\n"
    "        first, second = second, first + second\n"
    "    return result\n"
    "\n"
    "if __name__ == '__main__':\n"
    "    print(fib2(50))\n",
    encoding="utf-8",
)

fibo = importlib.import_module("fibo")
print("import 후 이름:", fibo.__name__)
print("Fibonacci:", fibo.fib2(30))

import 후 이름: fibo
Fibonacci: [0, 1, 1, 2, 3, 5, 8, 13, 21]


## 5. 모듈 검색 경로 `sys.path`

Python은 `sys.path`에 있는 디렉터리에서 import할 모듈을 찾습니다. 현재 실습에서는 임시 디렉터리를 검색 경로에 추가했습니다.

In [5]:
print("실습 디렉터리가 검색 경로에 포함됨:", str(lab_dir) in sys.path)
print("검색 경로 앞부분:")
for path in sys.path[:3]:
    print(" -", path)

실습 디렉터리가 검색 경로에 포함됨: True
검색 경로 앞부분:
 - C:\Users\juyoung\AppData\Local\Temp\chapter08_802mun4p
 - c:\Users\juyoung\AppData\Local\Programs\Python\Python313\python313.zip
 - c:\Users\juyoung\AppData\Local\Programs\Python\Python313\DLLs


## 6. 표준 모듈과 `dir()`

Python에 포함된 표준 모듈은 별도 설치 없이 사용할 수 있습니다. `dir()`과 `help()`로 모듈이 제공하는 이름을 탐색합니다.

In [6]:
import math
import random
from datetime import datetime

print("제곱근:", math.sqrt(16))
print("임의의 코인:", random.choice(["BTC", "ETH", "XRP"]))
print("현재 시각:", datetime.now())
print("math의 공개 이름 일부:", [name for name in dir(math) if not name.startswith("_")][:8])

제곱근: 4.0
임의의 코인: BTC
현재 시각: 2026-07-31 22:34:41.909038
math의 공개 이름 일부: ['acos', 'acosh', 'asin', 'asinh', 'atan', 'atan2', 'atanh', 'cbrt']


## 7. 패키지 만들기

패키지는 관련 모듈을 디렉터리 계층으로 묶습니다. `__init__.py`를 포함한 간단한 거래 패키지를 만들어 보겠습니다.

In [ ]:
import importlib

trading_dir = lab_dir / "trading"
trading_dir.mkdir()
(trading_dir / "__init__.py").write_text("PACKAGE_NAME = 'trading'\n", encoding="utf-8")
(trading_dir / "price.py").write_text(
    "def get_current_price(symbol):\n"
    "    prices = {'BTC': 105_000_000, 'ETH': 3_500_000}\n"
    "    return prices.get(symbol)\n",
    encoding="utf-8",
)

trading = importlib.import_module("trading")
price = importlib.import_module("trading.price")
print(price.get_current_price("BTC"))
print("패키지 이름:", trading.PACKAGE_NAME)

105000000
패키지 이름: trading


## 8. 절대 import와 상대 import

패키지의 최상위부터 경로를 쓰는 절대 import는 출처가 명확합니다. 패키지 내부에서는 점을 사용하는 상대 import도 사용할 수 있습니다.

In [ ]:
price = importlib.import_module("trading.price")
get_current_price = price.get_current_price

print(price.get_current_price("ETH"))
print(get_current_price("BTC"))

# 패키지 내부 모듈에서는 다음과 같은 상대 import를 사용할 수 있습니다.
relative_import_example = "from .. import price\nfrom . import volatility"
print(relative_import_example)

3500000
105000000
from .. import price
from . import volatility


## 9. `__all__`과 모듈 재사용

`__all__`은 패키지에서 공개할 이름을 문서화하는 데 사용할 수 있습니다. 일반적인 프로그램에서는 `import *`보다 필요한 이름을 명시하는 방식을 권장합니다.

In [ ]:
indicators_dir = trading_dir / "indicators"
indicators_dir.mkdir()
(indicators_dir / "__init__.py").write_text("__all__ = ['moving_average']\n", encoding="utf-8")
(indicators_dir / "moving_average.py").write_text(
    "def calculate(values):\n"
    "    return sum(values) / len(values)\n"
    "",
    encoding="utf-8",
)

moving_average = importlib.import_module("trading.indicators.moving_average")
calculate = moving_average.calculate
print("이동평균:", calculate([100, 105, 110]))

이동평균: 105.0


## 10. 변환 결과 검증

실습에 필요한 원본 문서와 노트북 경로, 모듈·패키지의 핵심 동작을 확인합니다.

In [10]:
workspace_dir = Path.cwd()
while workspace_dir != workspace_dir.parent and not (workspace_dir / "doc").exists():
    workspace_dir = workspace_dir.parent

source_path = workspace_dir / "doc" / "Chapter 08. 파이썬 모듈과 패키지.md"
assert source_path.exists(), source_path
assert coin_utils.calculate_return(100, 110) == 0.1
assert fibo.fib2(10) == [0, 1, 1, 2, 3, 5, 8]
assert get_current_price("BTC") == 105_000_000
assert calculate([100, 110]) == 105.0
print("Chapter 08 실습 검증 통과")
print("원본 문서:", source_path)
print("생성된 실습 디렉터리:", lab_dir)

Chapter 08 실습 검증 통과
원본 문서: c:\Users\juyoung\OneDrive\바탕 화면\python\python-edu-coin-trading\doc\Chapter 08. 파이썬 모듈과 패키지.md
생성된 실습 디렉터리: C:\Users\juyoung\AppData\Local\Temp\chapter08_802mun4p


## 실습 과제

1. `coin_utils.py`에 수수료를 제외한 순수익 계산 함수를 추가하세요.
2. `market_data.py`, `strategy.py`, `main.py`로 매매 신호 프로그램을 나누어 보세요.
3. `trading` 패키지에 `order.py` 모듈을 추가하고 주문 정보를 반환하세요.
4. 모듈을 직접 실행할 때만 동작하는 `if __name__ == "__main__":` 테스트를 추가하세요.
5. `sys.path`의 각 경로가 어떤 역할을 하는지 확인하세요.